In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime

# =========================================================
# 1) Load Shipment Data
# =========================================================
shipment = pd.read_csv("shipments.csv", parse_dates=["ship_date", "estimated_delivery", "actual_delivery"])

# =========================================================
# 2) Generate fulfillment_error_label.csv
# =========================================================

def assign_error_class(row):
    if row["delay_reason_code"] == "lost":
        return 3, "lost_package"
    
    if row["delay_reason_code"] in ["customer_not_home", "address_issue"] \
       or row["failed_attempts"] > 0:
        return 2, "failed_delivery"
    
    if row["delivery_delay_flag"] == 1:
        return 1, "late_delivery"
    
    return 0, "no_error"


labels = shipment.apply(assign_error_class, axis=1)
shipment["error_class"] = [x[0] for x in labels]
shipment["error_class_desc"] = [x[1] for x in labels]

# Select fields for label table
fulfillment_error_label = shipment[[
    "shipment_id", "order_id", "error_class", "error_class_desc",
    "delay_reason_code", "failed_attempts", "delivery_delay_flag"
]]

fulfillment_error_label.to_csv("fulfillment_error_label.csv", index=False)
print("Saved: fulfillment_error_label.csv")


# =========================================================
# 3) Generate dim_environment.csv
# =========================================================

# Date range
start_date = datetime(2023, 12, 3)
end_date = datetime(2025, 12, 2)
date_range = pd.date_range(start=start_date, end=end_date, freq="D")

# Warehouse list (regions)
warehouse_ids = ["EWR4", "SFO5", "ORD9", "PHX3", "ONT8", "SEA1", "DFW7"]

env_rows = []

for date in date_range:
    for wh in warehouse_ids:

        # --- Weather synthetic ---
        precipitation = max(0, np.random.normal(loc=2.0, scale=5.0))   # mm
        wind_speed = max(0, np.random.normal(loc=15, scale=10))        # km/h

        # --- Traffic synthetic ---
        traffic = min(max(np.random.normal(loc=4, scale=2), 0), 10)    # score 0-10

        # --- Holiday synthetic ---
        is_holiday = 1 if date.month == 12 and date.day in [24, 25, 31] else 0

        env_rows.append([
            date.strftime("%Y-%m-%d"),
            wh,
            precipitation,
            wind_speed,
            traffic,
            is_holiday
        ])

dim_environment = pd.DataFrame(env_rows, columns=[
    "date_key", "region", "precipitation_mm", "wind_speed_kmh",
    "traffic_congestion_score", "is_holiday"
])

dim_environment.to_csv("dim_environment.csv", index=False)
print("Saved: dim_environment.csv")


# =========================================================
# 4) Generate dim_operational.csv
# =========================================================

carriers = ["DHL", "UPS", "Amazon Logistics", "USPS", "FedEx"]

# Carrier reliability synthetic
carrier_rows = []
for carrier in carriers:
    delay_rate = np.round(np.random.uniform(0.05, 0.20), 3)
    fail_rate = np.round(np.random.uniform(0.01, 0.10), 3)
    lost_rate = np.round(np.random.uniform(0.001, 0.02), 3)

    carrier_rows.append([carrier, delay_rate, fail_rate, lost_rate])

carrier_df = pd.DataFrame(carrier_rows, columns=[
    "carrier", "carrier_delay_rate_30d", "carrier_fail_rate_30d", "carrier_lost_rate_30d"
])


# Warehouse load + route difficulty
warehouse_rows = []
warehouse_difficulty_map = {
    "EWR4": 1, "SFO5": 2, "ORD9": 0, "PHX3": 1,
    "ONT8": 2, "SEA1": 1, "DFW7": 0
}

for wh in warehouse_ids:

    # Synthetic load index (0.3 ~ 0.95)
    load_index = np.round(np.random.uniform(0.3, 0.95), 3)

    # Route difficulty = base(distance) + region weight + noise
    # But distance is shipment-level, so here we define only region difficulty baseline (0-2)

    route_difficulty = warehouse_difficulty_map[wh] + random.choice([0, 1])

    warehouse_rows.append([wh, load_index, route_difficulty])

warehouse_df = pd.DataFrame(warehouse_rows, columns=[
    "warehouse_id", "warehouse_load_index", "route_difficulty_score"
])


# Merge carrier + warehouse into dim_operational
dim_operational = carrier_df.merge(
    warehouse_df.assign(key=1),
    how="cross"
).drop(columns="key")

dim_operational.to_csv("dim_operational.csv", index=False)
print("Saved: dim_operational.csv")


Saved: fulfillment_error_label.csv
Saved: dim_environment.csv
Saved: dim_operational.csv
